# Session 6: ANOVA and Linear Regression

**Module 3: Programming for Biological Data**  
**Date:** January 21, 2026 | 18:30 – 21:30  
**Instructor:** Dr. Haogao Gu

---

## Learning Objectives

By the end of this session, you will be able to:
1. Compare means across **multiple groups** using ANOVA
2. Identify which specific groups differ with **post-hoc tests**
3. Build **linear regression models** for calibration curves
4. Assess model fit using **R²** and residual analysis

---

## Clinical Context

In medical laboratories:
- **ANOVA:** Compare performance across multiple analyzers, reagent lots, or operators
- **Regression:** Build and verify calibration curves (CLSI EP06)

In [ ]:
# ============================================
# STANDARD SETUP PROTOCOL
# Run this cell first in every session
# ============================================

options(repos = c(CRAN = "https://cloud.r-project.org"))

if (!requireNamespace("BiocManager", quietly = TRUE))
  install.packages("BiocManager")

ensure_loaded <- function(pkg) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    message(paste("Installing", pkg, "..."))
    BiocManager::install(pkg, update = FALSE, ask = FALSE)
  }
  library(pkg, character.only = TRUE)
}

ensure_loaded("ggplot2")

cat("✅ Setup complete! Ready for Session 6.")

---

# Part 1: Analysis of Variance (ANOVA)

## 40 minutes

---

## 1.1 When to Use ANOVA

**T-test:** Compare **2 groups**  
**ANOVA:** Compare **3 or more groups**

### Clinical Scenario

Your lab has **three chemistry analyzers** (A, B, C). You run the same QC material on each.

**Question:** Are they giving consistent results, or is one analyzer biased?

In [ ]:
# Create analyzer comparison data
# 10 measurements per analyzer

set.seed(2024)

analyzer_data <- data.frame(
  Analyzer = rep(c("A", "B", "C"), each = 10),
  Glucose = c(
    # Analyzer A: centered around 102
    round(rnorm(10, mean = 102, sd = 1.5), 0),
    # Analyzer B: centered around 98 (slight negative bias)
    round(rnorm(10, mean = 98, sd = 1.5), 0),
    # Analyzer C: centered around 106 (positive bias)
    round(rnorm(10, mean = 106, sd = 1.5), 0)
  )
)

head(analyzer_data, 15)

In [ ]:
# Summary by analyzer
aggregate(Glucose ~ Analyzer, data = analyzer_data, 
          FUN = function(x) c(Mean = mean(x), SD = sd(x)))

## 1.2 Visualizing Group Differences

In [ ]:
# Boxplot comparison
boxplot(Glucose ~ Analyzer, data = analyzer_data,
        main = "Glucose QC by Analyzer",
        xlab = "Analyzer",
        ylab = "Glucose (mg/dL)",
        col = c("coral", "steelblue", "seagreen"))

# Add reference line at target (100 mg/dL)
abline(h = 100, col = "red", lty = 2, lwd = 2)

## 1.3 The Logic of ANOVA

ANOVA asks: **Is variance BETWEEN groups larger than variance WITHIN groups?**

### The F-statistic

$$F = \frac{\text{Variance Between Groups}}{\text{Variance Within Groups}}$$

- **Large F** → Groups are different (between > within)
- **Small F** → Groups are similar (within variance dominates)

### Hypotheses

- **H₀:** All group means are equal (μA = μB = μC)
- **H₁:** At least one group mean differs

In [ ]:
# Perform One-Way ANOVA
anova_result <- aov(Glucose ~ Analyzer, data = analyzer_data)

# View the ANOVA table
summary(anova_result)

### Interpreting the ANOVA Table

| Column | Meaning |
|--------|--------|
| Df | Degrees of freedom |
| Sum Sq | Sum of squares (variance) |
| Mean Sq | Mean square (Sum Sq / Df) |
| F value | Test statistic |
| Pr(>F) | **P-value** |

**If p < 0.05:** At least one analyzer differs significantly

In [ ]:
# Extract p-value
p_value <- summary(anova_result)[[1]][["Pr(>F)"]][1]

if (p_value < 0.05) {
  cat(sprintf("⚠️ Significant difference detected! (p = %.4f)\n", p_value))
  cat("At least one analyzer differs from the others.\n")
} else {
  cat(sprintf("✅ No significant difference (p = %.4f)\n", p_value))
  cat("All analyzers are performing consistently.\n")
}

## 1.4 Post-Hoc Testing: Which Analyzer is Different?

ANOVA tells us **something is different**, but not **what**.

**Tukey's Honest Significant Difference (HSD)** performs pairwise comparisons.

In [ ]:
# Tukey HSD post-hoc test
tukey_result <- TukeyHSD(anova_result)
print(tukey_result)

In [ ]:
# Visualize pairwise comparisons
plot(tukey_result, las = 1)
abline(v = 0, col = "red", lty = 2)

### Reading Tukey Results

| Column | Meaning |
|--------|--------|
| diff | Difference between group means |
| lwr, upr | 95% confidence interval |
| p adj | Adjusted p-value |

**If the confidence interval crosses zero OR p adj > 0.05:** No significant difference  
**If the interval does NOT cross zero AND p adj < 0.05:** Significant difference

In [ ]:
# Clinical interpretation
cat("\n=== Clinical Action Items ===\n\n")

tukey_df <- as.data.frame(tukey_result$Analyzer)
for (i in 1:nrow(tukey_df)) {
  comparison <- rownames(tukey_df)[i]
  p_adj <- tukey_df$`p adj`[i]
  diff <- tukey_df$diff[i]
  
  if (p_adj < 0.05) {
    cat(sprintf("⚠️ %s: Significant difference (%.1f mg/dL, p = %.4f)\n", 
                comparison, diff, p_adj))
  } else {
    cat(sprintf("✅ %s: No significant difference (p = %.4f)\n", 
                comparison, p_adj))
  }
}

## 1.5 ANOVA Assumptions

Before trusting ANOVA results, check:

1. **Normality** within each group (Shapiro-Wilk)
2. **Homogeneity of variances** (Bartlett or Levene test)
3. **Independence** of observations

In [ ]:
# Check normality of residuals
shapiro.test(residuals(anova_result))

# Check homogeneity of variances
bartlett.test(Glucose ~ Analyzer, data = analyzer_data)

---

# Part 2: Linear Regression & Calibration

## 40 minutes

---

## 2.1 The Standard Curve

Every quantitative assay relies on a **calibration curve**:

$$\text{Signal} = \text{Slope} \times \text{Concentration} + \text{Intercept}$$

Or in math notation: $y = mx + c$

### Clinical Application: CLSI EP06

CLSI EP06 defines **linearity verification**:
- Test samples across the reportable range
- Verify that the assay response is linear
- Check for saturation or non-linearity at extremes

In [ ]:
# Create linearity verification data
# 6 concentrations, measured in triplicate

linearity_data <- data.frame(
  Concentration = rep(c(0, 25, 50, 100, 200, 400), each = 3),
  Signal = c(
    0.052, 0.048, 0.055,   # 0 (blank)
    0.245, 0.251, 0.238,   # 25
    0.478, 0.485, 0.472,   # 50
    0.952, 0.965, 0.948,   # 100
    1.875, 1.892, 1.868,   # 200
    3.425, 3.458, 3.412    # 400
  )
)

print(linearity_data)

In [ ]:
# Visualize the raw data
plot(linearity_data$Concentration, linearity_data$Signal,
     main = "Linearity Verification: Raw Data",
     xlab = "Concentration (mg/dL)",
     ylab = "Signal (OD)",
     pch = 19, col = "steelblue", cex = 1.5)

## 2.2 Fitting a Linear Model

The `lm()` function fits a linear model in R:

```r
model <- lm(Response ~ Predictor, data = dataset)
```

In our case: `Signal ~ Concentration`

In [ ]:
# Fit linear model
model <- lm(Signal ~ Concentration, data = linearity_data)

# View summary
summary(model)

### Interpreting the Output

| Term | Meaning |
|------|--------|
| (Intercept) | The y-intercept (signal at conc = 0) |
| Concentration | The slope (signal change per unit conc) |
| Std. Error | Uncertainty in estimates |
| t value, Pr(>|t|) | Significance of each term |
| **Multiple R-squared** | Proportion of variance explained |
| **Adjusted R-squared** | R² adjusted for number of predictors |

In [ ]:
# Extract key values
intercept <- coef(model)[1]
slope <- coef(model)[2]
r_squared <- summary(model)$r.squared

cat("=== Calibration Curve Parameters ===\n\n")
cat(sprintf("Equation: Signal = %.4f × Concentration + %.4f\n", slope, intercept))
cat(sprintf("R² = %.4f (%.1f%% variance explained)\n", r_squared, r_squared * 100))

## 2.3 Plotting the Regression Line

In [ ]:
# Plot data with regression line
plot(linearity_data$Concentration, linearity_data$Signal,
     main = "Calibration Curve with Best-Fit Line",
     xlab = "Concentration (mg/dL)",
     ylab = "Signal (OD)",
     pch = 19, col = "steelblue", cex = 1.5)

# Add regression line
abline(model, col = "red", lwd = 2)

# Add equation and R² to plot
legend("topleft", 
       legend = c(
         sprintf("y = %.4fx + %.4f", slope, intercept),
         sprintf("R² = %.4f", r_squared)
       ),
       bty = "n", cex = 1.2)

## 2.4 Using ggplot2 for Publication-Quality Plots

In [ ]:
# Professional calibration curve with ggplot2
ggplot(linearity_data, aes(x = Concentration, y = Signal)) +
  geom_point(size = 3, color = "steelblue") +
  geom_smooth(method = "lm", se = TRUE, color = "red", fill = "pink", alpha = 0.3) +
  labs(
    title = "Calibration Curve: Linearity Verification",
    subtitle = sprintf("y = %.4fx + %.4f, R² = %.4f", slope, intercept, r_squared),
    x = "Concentration (mg/dL)",
    y = "Signal (OD)"
  ) +
  theme_minimal() +
  theme(plot.title = element_text(hjust = 0.5, size = 14, face = "bold"),
        plot.subtitle = element_text(hjust = 0.5, size = 12))

## 2.5 Residual Analysis: Checking Model Fit

**Residuals** = Observed - Predicted

A good model has:
- Residuals randomly scattered around zero
- No patterns (e.g., curvature suggests non-linearity)
- Consistent spread (homoscedasticity)

In [ ]:
# Calculate residuals
linearity_data$Predicted <- predict(model)
linearity_data$Residual <- residuals(model)

# View
print(linearity_data)

In [ ]:
# Residual plot
plot(linearity_data$Concentration, linearity_data$Residual,
     main = "Residual Plot",
     xlab = "Concentration (mg/dL)",
     ylab = "Residual (Observed - Predicted)",
     pch = 19, col = "steelblue", cex = 1.5)

abline(h = 0, col = "red", lty = 2, lwd = 2)

# What to look for:
# - Random scatter: Good!
# - Curved pattern: Non-linearity (consider polynomial or log transform)
# - Funnel shape: Heteroscedasticity (consider weighted regression)

In [ ]:
# Diagnostic plots (built-in R function)
par(mfrow = c(2, 2))
plot(model)
par(mfrow = c(1, 1))

## 2.6 Predicting Unknown Concentrations

Once we have a calibration curve, we can use it to calculate unknown concentrations from measured signals.

In [ ]:
# Prediction: What concentration gives a signal of 1.5 OD?
# Rearrange: Concentration = (Signal - Intercept) / Slope

unknown_signal <- 1.5
predicted_conc <- (unknown_signal - intercept) / slope

cat(sprintf("Unknown sample signal: %.3f OD\n", unknown_signal))
cat(sprintf("Predicted concentration: %.1f mg/dL\n", predicted_conc))

In [ ]:
# Batch prediction
unknown_samples <- data.frame(
  SampleID = c("S1", "S2", "S3", "S4"),
  Signal = c(0.512, 1.234, 2.456, 3.100)
)

unknown_samples$Concentration <- (unknown_samples$Signal - intercept) / slope

print(unknown_samples)

## 2.7 Linearity Acceptance Criteria

### CLSI EP06 Criteria

| Criterion | Typical Requirement |
|-----------|--------------------|
| R² | ≥ 0.99 for most assays |
| Residuals | Within ±10% of expected |
| Recovery | 95–105% at each level |

**Note:** Specific requirements vary by analyte and method.

In [ ]:
# Linearity acceptance check
cat("=== Linearity Verification Report ===\n\n")

r2_threshold <- 0.99

if (r_squared >= r2_threshold) {
  cat(sprintf("✅ R² = %.4f (≥ %.2f) - PASS\n", r_squared, r2_threshold))
} else {
  cat(sprintf("⚠️ R² = %.4f (< %.2f) - FAIL\n", r_squared, r2_threshold))
}

# Check max residual as % of predicted
linearity_data$ResidualPct <- abs(linearity_data$Residual / linearity_data$Predicted) * 100
max_residual_pct <- max(linearity_data$ResidualPct, na.rm = TRUE)

if (max_residual_pct <= 10) {
  cat(sprintf("✅ Max residual = %.1f%% (≤ 10%%) - PASS\n", max_residual_pct))
} else {
  cat(sprintf("⚠️ Max residual = %.1f%% (> 10%%) - FAIL\n", max_residual_pct))
}

---

# Key Takeaways

1. **ANOVA** compares means across 3+ groups
   - `aov(y ~ group)` + `summary()`
   - Use `TukeyHSD()` for pairwise comparisons

2. **Linear regression** fits y = mx + c
   - `lm(y ~ x)` + `summary()`
   - R² indicates fit quality

3. **Residual analysis** validates model assumptions
   - Random scatter = good fit
   - Patterns = model problems

4. **Clinical context** is key
   - ANOVA → analyzer/lot comparisons
   - Regression → calibration curves (CLSI EP06)

---

## Now proceed to Tutorial 6! 🧪